In [ ]:
from datasets import load_dataset
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from collections import Counter, defaultdict
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. DATA PREPARATION (EVASION LABEL)
# ==========================================
print("Loading dataset...")
dataset = load_dataset("ailsntua/QEvasion")

# Filter training data: Remove entries where evasion_label is None or empty
# We DO NOT filter the test set, as we need to generate predictions for every row.
dataset['train'] = dataset['train'].filter(lambda x: x['evasion_label'] is not None and x['evasion_label'] != "")

# Prepare labels based on EVASION from the training set
labels = sorted(dataset['train'].unique('evasion_label'))
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

print(f"Labels mapped: {label2id}")

def preprocess_data(example):
    # --- FIX IS HERE ---
    # We get the label
    label = example.get('evasion_label')

    # We only try to map it if it is NOT None AND NOT an empty string
    if label is not None and label != "":
        example['labels'] = label2id[label]

    return example

dataset = dataset.map(preprocess_data)

# Tokenizer setup
model_checkpoint = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

print("Tokenizing...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# ==========================================
# 2. CLASS WEIGHTS & FOCAL LOSS
# ==========================================
def get_class_weights(dataset_split, num_labels):
    label_counts = Counter(dataset_split["labels"])
    total_samples = len(dataset_split)

    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 0)
        if count == 0: count = 1 # Prevent div by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    return torch.tensor(class_weights, dtype=torch.float32, device=device)

class_weights = get_class_weights(tokenized_datasets['train'], num_labels)
print(f"Class weights: {class_weights}")

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# ==========================================
# 3. HYBRID CUSTOM TRAINER
# ==========================================
class HybridTrainer(Trainer):
    """
    Combines Focal Loss logic with Multi-Annotator Evaluation logic.
    """
    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    # Custom Loss (Focal)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

    # Custom Evaluation (Multi-Annotator F1 Macro)
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset

        # 1. Run prediction
        output = self.predict(eval_dataset, metric_key_prefix="test")
        preds = np.argmax(output.predictions, axis=1)
        pred_labels = [self.model.config.id2label[p] for p in preds]

        # 2. Access raw dataset to get annotator columns
        # Note: We rely on the fact that eval_dataset preserves the original dataset rows
        hf_test_data = eval_dataset

        tp = defaultdict(int)
        fp = defaultdict(int)
        fn = defaultdict(int)
        all_classes = set()

        # 3. Calculate metrics based on "Match Any Annotator" logic
        for i, pred_label in enumerate(pred_labels):
            # Safe access to annotators
            anns = [
                hf_test_data[i]['annotator1'],
                hf_test_data[i]['annotator2'],
                hf_test_data[i]['annotator3']
            ]
            true_set = set([a for a in anns if a])

            all_classes.add(pred_label)
            all_classes.update(true_set)

            if pred_label in true_set:
                tp[pred_label] += 1
            else:
                fp[pred_label] += 1
                for true_cls in true_set:
                    fn[true_cls] += 1

        # 4. Compute Macro F1
        f1_scores = []
        for cls in all_classes:
            class_tp = tp[cls]
            class_fp = fp[cls]
            class_fn = fn[cls]

            prec = class_tp / (class_tp + class_fp) if (class_tp + class_fp) > 0 else 0.0
            rec = class_tp / (class_tp + class_fn) if (class_tp + class_fn) > 0 else 0.0
            f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
            f1_scores.append(f1)

        macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

        metrics = {f"{metric_key_prefix}_f1_macro": macro_f1}
        self.log(metrics)
        return metrics

# ==========================================
# 4. TRAINING SETUP
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="ModernBERT_Evasion_Hybrid",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
)

trainer = HybridTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    class_weights=class_weights,
    focal_gamma=1.0,
)

print("Starting training...")
trainer.train()
print("Training completed!")

# ==========================================
# 5. FINAL REPORT & PREDICTION FILE
# ==========================================

print("\n" + "="*60)
print("FINAL TEST REPORT (Using Multi-Annotator Logic)")
print("="*60)

# 1. Get Predictions from Best Model
output = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(output.predictions, axis=1)
pred_labels = [id2label[p] for p in preds]

# 2. Print Classification Report
hf_test_data = tokenized_datasets["test"]
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)
all_classes = set()

for i, pred_label in enumerate(pred_labels):
    anns = [
        hf_test_data[i]['annotator1'],
        hf_test_data[i]['annotator2'],
        hf_test_data[i]['annotator3']
    ]
    true_set = set([a for a in anns if a])

    all_classes.add(pred_label)
    all_classes.update(true_set)

    if pred_label in true_set:
        tp[pred_label] += 1
    else:
        fp[pred_label] += 1
        for true_cls in true_set:
            fn[true_cls] += 1

f1_scores = []
all_classes_sorted = sorted(list(all_classes))

col_1_width = max(max(len(str(cls)) for cls in all_classes_sorted), len("Class")) + 2
header = (
    f"{'Class':<{col_1_width}} | {'TP':<5} | {'FP':<5} | {'FN':<5} | "
    f"{'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}"
)

print(header)
print("-" * len(header))

for cls in all_classes_sorted:
    class_tp = tp[cls]
    class_fp = fp[cls]
    class_fn = fn[cls]

    prec = class_tp / (class_tp + class_fp) if (class_tp + class_fp) > 0 else 0.0
    rec = class_tp / (class_tp + class_fn) if (class_tp + class_fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    f1_scores.append(f1)

    print(
        f"{cls:<{col_1_width}} | {class_tp:<5} | {class_fp:<5} | {class_fn:<5} | "
        f"{prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f}"
    )

macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

print("-" * len(header))
print(f"FINAL MACRO F1 SCORE: {macro_f1:.6f}")
print("="*len(header))

# 3. Save Model
trainer.save_model("ModernBERT_Evasion_Best")
tokenizer.save_pretrained("ModernBERT_Evasion_Best")
print("Model saved.")

# ==========================================
# 6. GENERATE PREDICTION FILE
# ==========================================
print("\nGenerating prediction file for submission...")

output_file = "prediction" # No extension

with open(output_file, "w") as f:
    for label in pred_labels:
        f.write(f"{label}\n")

print(f"✅ Successfully saved {len(pred_labels)} labels to the file '{output_file}'")

Using device: cuda
Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-360106240.py:100: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `HybridTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting training...


Epoch,Training Loss,Validation Loss


Training completed!

FINAL TEST REPORT (Using Multi-Annotator Logic)


Class                 | TP    | FP    | FN    | Precision  | Recall     | F1-Score  
------------------------------------------------------------------------------------
Claims ignorance      | 9     | 4     | 4     | 0.6923     | 0.6923     | 0.6923    
Clarification         | 3     | 0     | 1     | 1.0000     | 0.7500     | 0.8571    
Declining to answer   | 4     | 12    | 9     | 0.2500     | 0.3077     | 0.2759    
Deflection            | 9     | 19    | 30    | 0.3214     | 0.2308     | 0.2687    
Dodging               | 12    | 15    | 66    | 0.4444     | 0.1538     | 0.2286    
Explicit              | 37    | 40    | 53    | 0.4805     | 0.4111     | 0.4431    
General               | 28    | 30    | 60    | 0.4828     | 0.3182     | 0.3836    
Implicit              | 29    | 55    | 49    | 0.3452     | 0.3718     | 0.3580    
Partial/half-answer   | 0     | 2     | 9     | 0.0000     | 0.0000     | 0.0000    
-----------------------------------------------------------------